In [ ]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/faster_rcnn
%cp VOC2007.zip /content
%cp VOC2012.zip /content
%cd /content

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/faster_rcnn
^C


^C
/content


In [2]:
from pathlib import Path
import zipfile

data_path = Path("data/")
data_path.mkdir(exist_ok=True)

voc2007_zip_path = Path("VOC2007.zip")
voc2012_zip_path = Path("VOC2012.zip")

if not voc2007_zip_path.exists() or not voc2012_zip_path.exists():
    raise RuntimeError("Dataset not found.")

print("Extracting 2007 dataset ...")

with zipfile.ZipFile(voc2007_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

print(f"Extracting 2012 dataset ...")
with zipfile.ZipFile(voc2012_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

Extracting 2007 dataset ...
Extracting 2012 dataset ...


In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
from src.backbone import Backbone
from src.rpn import RPN_Head, RegionProposalNetwork

backbone = Backbone().to(device)
rpn_head = RPN_Head(in_channels=1024, mid_channels=512).to(device)

In [3]:
from pathlib import Path
import torch

checkpoint_dir = Path("/content/drive/MyDrive/faster_rcnn/checkpoints")
# checkpoint_dir = Path("checkpoints")

existing_checkpoints = sorted(checkpoint_dir.glob("step1_epoch_*.pt"),
                              key = lambda p : int(p.stem.split("_epoch_")[1]))

print(f"Existing checkpoints: {existing_checkpoints}")

if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")

    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    backbone.load_state_dict(checkpoint['backbone_state_dict'])
    rpn_head.load_state_dict(checkpoint['rpn_head_state_dict'])



Existing checkpoints: [PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_1.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_2.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_3.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_4.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_5.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_6.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_7.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_8.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_9.pt'), PosixPath('/content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_10.pt')]
Loading checkpoint: /content/drive/MyDrive/faster_rcnn/checkpoints/step1_epoch_10.pt


In [4]:
from src.backbone import backbone_transform
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

transform = backbone_transform

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

combined_train_img_paths = voc2007_img_paths_train + voc2012_img_paths_train

train_dataloader = create_voc_dataloader(img_paths_list=combined_train_img_paths, transform=transform, batch_size=2, shuffle=True)

In [5]:
batch_imgs, batch_boxes, batch_labels, img_sizes_before_pad = next(iter(train_dataloader))

rpn_network = RegionProposalNetwork(rpn_head=rpn_head).to(device)

with torch.inference_mode():
    backbone.eval()
    rpn_head.eval()
    rpn_network.eval()

    batch_feature_maps = backbone(batch_imgs.to(device))
    print(f"batch_feature_maps.shape: {batch_feature_maps.shape}")
    batch_scores, batch_proposals = rpn_network(batch_feature_maps, batch_imgs.shape[2], batch_imgs.shape[3], img_sizes_before_pad)
    

batch_feature_maps.shape: torch.Size([2, 1024, 57, 38])
torch.Size([6000, 4])
torch.Size([262, 4])
torch.Size([262, 4])
torch.Size([6000, 4])
torch.Size([266, 4])
torch.Size([266, 4])


In [6]:
batch_proposals[0].shape

torch.Size([181, 4])

In [7]:
import torch.nn as nn

class RoIPool(nn.Module):
    def __init__(self, output_size, stride):
        self.output_size = output_size
        self.stride = stride

    def forward(self, feature_maps, batch_proposals):
        return self._project_to_feature_map(batch_proposals, feature_maps.shape[2], feature_maps.shape[3])

    def _project_to_feature_map(self, proposals, feat_height, feat_width):
        x1 = proposals[:,:, 0]
        y1 = proposals[:,:, 1]
        x2 = proposals[:,:, 2]
        y2 = proposals[:,:, 3]

        fx1 = torch.round(x1 / self.stride)
        fy1 = torch.round(y1 / self.stride)
        fx2 = torch.round(x2 / self.stride)
        fy2 = torch.round(y2 / self.stride)

        # Convert to long type for holding larger values and prevent overflow due to decimal double
        fx1 = fx1.long()
        fy1 = fy1.long()
        fx2 = fx2.long()
        fy2 = fy2.long()

        fx1 = torch.clamp(fx1, 0, feat_width - 1)
        fx2 = torch.clamp(fx2, 0, feat_width - 1)
        fy1 = torch.clamp(fy1, 0, feat_height - 1)
        fy2 = torch.clamp(fy2, 0, feat_height - 1)

        return torch.stack([fx1, fy1, fx2, fy2], dim=-1)  # [B, N, 4]